# Proof-of-Concept: Connect c-log-analysis to Sentry for Automated Issue Creation

This notebook demonstrates how to:
1. Simulate anomaly detection output from c-log-analysis
2. Create an issue in Sentry when a deviation is detected using Sentry MCP tools
3. Bridge c-log-analysis with Sentry for L2P's bug-demo/Cline workflow

## Step 1: Install Dependencies

Install required Python packages for this proof of concept.

In [1]:
# Install dependencies if needed
import subprocess
import sys

def install_package(package):
    try:
        __import__(package)
        print(f"{package} is already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Install required packages
install_package("requests")
install_package("json")

import os
import requests
import json
from datetime import datetime
import uuid

requests is already installed
json is already installed


## Step 2: Simulate c-log-analysis Output

Here, we'll simulate test data as if it came from c-log-analysis. In practice, you would parse this from your pipeline output.

In [2]:
# Example simulated output from c-log-analysis
block_id = 42
similarity_score = 0.60  # Simulated anomaly (below threshold)
threshold = 0.85
log_excerpt = "[2025-08-06 14:54:00] ERROR: Unexpected shutdown in service X. Memory usage exceeded 95%."
service_name = "payment-processor"
pod_name = "payment-processor-7d8f9b-xyz123"

if similarity_score < threshold:
    detected = True
    deviation_severity = "high" if similarity_score < 0.5 else "medium"
    print(f"🚨 Deviation detected in block {block_id} (score: {similarity_score})")
    print(f"   Severity: {deviation_severity}")
    print(f"   Service: {service_name}")
    print(f"   Pod: {pod_name}")
else:
    detected = False
    print(f"✅ No deviation detected (score: {similarity_score})")

🚨 Deviation detected in block 42 (score: 0.6)
   Severity: medium
   Service: payment-processor
   Pod: payment-processor-7d8f9b-xyz123


## Step 3: Configure Sentry Connection

Instead of using direct API calls, we'll demonstrate how this would work with the Sentry MCP server that Cline uses.

In [3]:
# Sentry configuration - these match the MCP server setup
SENTRY_ORG = "cisco-og"
SENTRY_PROJECT = "bug-demo"
SENTRY_REGION_URL = "https://us.sentry.io"

print(f"Sentry Configuration:")
print(f"  Organization: {SENTRY_ORG}")
print(f"  Project: {SENTRY_PROJECT}")
print(f"  Region: {SENTRY_REGION_URL}")

Sentry Configuration:
  Organization: cisco-og
  Project: bug-demo
  Region: https://us.sentry.io


In [5]:
!pip install sentry-sdk
import sentry_sdk

sentry_sdk.init(
    dsn="https://66f1182e2b7fdc0a34150100e3d0c6a3@o4509759301877760.ingest.us.sentry.io/4509799280410624",
    traces_sample_rate=1.0,
    #environment="development",  # or "production"
)

try:
   foo()  # This will raise NameError since foo is not defined
except Exception as e:
    sentry_sdk.capture_exception(e)

  Using cached sentry_sdk-2.34.1-py2.py3-none-any.whl.metadata (10 kB)
Using cached sentry_sdk-2.34.1-py2.py3-none-any.whl (357 kB)

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


## Step 4: Create Sentry Event Data

Prepare the event data that would be sent to Sentry. In the actual implementation, Cline would use the Sentry MCP tools to create issues.

In [5]:
def create_sentry_event_data(block_id, similarity_score, threshold, log_excerpt, service_name, pod_name):
    """
    Create structured event data for Sentry.
    This simulates what would be sent via the Sentry MCP tools.
    """
    event_id = str(uuid.uuid4()).replace('-', '')
    timestamp = datetime.utcnow().isoformat() + 'Z'
    
    # Determine severity level
    if similarity_score < 0.3:
        level = "fatal"
    elif similarity_score < 0.5:
        level = "error"
    elif similarity_score < 0.7:
        level = "warning"
    else:
        level = "info"
    
    event_data = {
        "event_id": event_id,
        "timestamp": timestamp,
        "level": level,
        "message": f"Log anomaly detected in {service_name} (block {block_id})",
        "logger": "c-log-analysis",
        "platform": "python",
        "tags": {
            "system": "log-analysis",
            "service": service_name,
            "pod": pod_name,
            "automated": "true",
            "anomaly_type": "similarity_deviation",
            "severity": level
        },
        "extra": {
            "block_id": block_id,
            "similarity_score": similarity_score,
            "threshold": threshold,
            "deviation_amount": threshold - similarity_score,
            "log_excerpt": log_excerpt,
            "source_system": "c-log-analysis",
            "analysis_timestamp": timestamp,
            "kubernetes_pod": pod_name,
            "service_name": service_name
        },
        "contexts": {
            "runtime": {
                "name": "c-log-analysis",
                "version": "1.0.0"
            },
            "os": {
                "name": "kubernetes"
            }
        },
        "fingerprint": [
            "c-log-analysis",
            service_name,
            "similarity-anomaly"
        ]
    }
    
    return event_data

# Create event data if anomaly detected
if detected:
    event_data = create_sentry_event_data(
        block_id, similarity_score, threshold, 
        log_excerpt, service_name, pod_name
    )
    
    print("\n📋 Sentry Event Data Created:")
    print(f"  Event ID: {event_data['event_id']}")
    print(f"  Level: {event_data['level']}")
    print(f"  Message: {event_data['message']}")
    print(f"  Tags: {event_data['tags']}")
    print(f"  Deviation: {event_data['extra']['deviation_amount']:.2f} below threshold")
else:
    print("\n✅ No event data created - no anomaly detected")


📋 Sentry Event Data Created:
  Event ID: b64875ba1bf74bc085f5107532274be9
  Level: warning
  Message: Log anomaly detected in payment-processor (block 42)
  Tags: {'system': 'log-analysis', 'service': 'payment-processor', 'pod': 'payment-processor-7d8f9b-xyz123', 'automated': 'true', 'anomaly_type': 'similarity_deviation', 'severity': 'warning'}
  Deviation: 0.25 below threshold


/var/folders/kk/lc20xhqn4fv20r5zj6bdv4d00000gn/T/ipykernel_37370/902012374.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat() + 'Z'


## Step 5: Send Anomaly Event to Sentry

Now that Sentry integration is working, we can send structured anomaly events directly to Sentry using the Python SDK. If an anomaly is detected, the event data is attached as tags and extras for rich context.


In [9]:
if detected:
    try:
        raise RuntimeError(f"Log anomaly detected in {service_name} (block {block_id}): {log_excerpt}")
    except Exception as e:
        with sentry_sdk.push_scope() as scope:
            scope.set_tag("service", service_name)
            scope.set_tag("pod", pod_name)
            scope.set_tag("severity", deviation_severity)
            scope.set_extra("block_id", block_id)
            scope.set_extra("similarity_score", similarity_score)
            scope.set_extra("threshold", threshold)
            scope.set_extra("deviation_amount", threshold - similarity_score)
            scope.set_extra("log_excerpt", log_excerpt)
            scope.set_extra("analysis_timestamp", datetime.utcnow().isoformat() + 'Z')
            sentry_sdk.capture_exception(e)
    print("\n✅ Sentry event sent for anomaly.")
else:
    print("\n✅ No anomaly detected; no Sentry event sent.")


✅ Sentry event sent for anomaly.


/var/folders/kk/lc20xhqn4fv20r5zj6bdv4d00000gn/T/ipykernel_37370/60803078.py:5: DeprecationWarning: sentry_sdk.push_scope is deprecated and will be removed in the next major version. Please consult our migration guide to learn how to migrate to the new API: https://docs.sentry.io/platforms/python/migration/1.x-to-2.x#scope-pushing
  with sentry_sdk.push_scope() as scope:
/var/folders/kk/lc20xhqn4fv20r5zj6bdv4d00000gn/T/ipykernel_37370/60803078.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  scope.set_extra("analysis_timestamp", datetime.utcnow().isoformat() + 'Z')


## Step 6: Complete Workflow Simulation

Demonstrate the complete workflow from c-log-analysis to Sentry to GitHub issue creation.

In [7]:
def simulate_complete_workflow_with_cline():
    print("\n🔄 Complete L2P Automated Workflow with Cline:")
    print("\n" + "="*60)
    if detected:
        print("\n1. 📊 c-log-analysis detects anomaly")
        print(f"   - Block ID: {block_id}")
        print(f"   - Similarity Score: {similarity_score} (below threshold {threshold})")
        print(f"   - Service: {service_name}")

        print("\n2. 🕵️‍♂️ Cline fetches the latest issue from Sentry")
        print("   - cline sentry issues:list --project bug-demo --org cisco-og --limit 1")

        print("\n3. 📝 Cline creates a new GitHub issue linked to the Sentry issue")
        print("   - cline github issues:create --repo ccapetz/L2P --title 'Log anomaly detected in {service_name}' --body 'See Sentry issue: <SENTRY_ISSUE_URL>' --labels bug,automated,sentry-integration")

        print("\n4. 🛠️ Fix the bug based on the Sentry issue")
        print("   - Implement code changes as needed")

        print("\n5. 💾 Commit changes in a new branch referencing both issues")
        print("   - git checkout -b fix/{block_id}-anomaly")
        print("   - git commit -am 'Fix: {service_name} anomaly (Sentry: <SENTRY_ISSUE_ID>, GitHub: <GITHUB_ISSUE_ID>)'")

        print("\n6. 🚀 Push new branch to GitHub")
        print("   - git push origin fix/{block_id}-anomaly")

        print("\n7. 🔗 Open a PR referencing the GitHub issue")
        print("   - cline github pr:create --repo ccapetz/L2P --title 'Fix {service_name} anomaly' --body 'Closes <GITHUB_ISSUE_URL>' --base main --head fix/{block_id}-anomaly")

    else:
        print("\n✅ No workflow needed - system operating normally")
        print(f"   - Similarity score {similarity_score} above threshold {threshold}")
        print("   - Continuing monitoring...")

    print("\n" + "="*60)
    print("\n🎯 Workflow Complete!")

simulate_complete_workflow_with_cline()


🔄 Complete L2P Automated Workflow with Cline:


1. 📊 c-log-analysis detects anomaly
   - Block ID: 42
   - Similarity Score: 0.6 (below threshold 0.85)
   - Service: payment-processor

2. 🕵️‍♂️ Cline fetches the latest issue from Sentry
   - cline sentry issues:list --project bug-demo --org cisco-og --limit 1

3. 📝 Cline creates a new GitHub issue linked to the Sentry issue
   - cline github issues:create --repo ccapetz/L2P --title 'Log anomaly detected in {service_name}' --body 'See Sentry issue: <SENTRY_ISSUE_URL>' --labels bug,automated,sentry-integration

4. 🛠️ Fix the bug based on the Sentry issue
   - Implement code changes as needed

5. 💾 Commit changes in a new branch referencing both issues
   - git checkout -b fix/{block_id}-anomaly
   - git commit -am 'Fix: {service_name} anomaly (Sentry: <SENTRY_ISSUE_ID>, GitHub: <GITHUB_ISSUE_ID>)'

6. 🚀 Push new branch to GitHub
   - git push origin fix/{block_id}-anomaly

7. 🔗 Open a PR referencing the GitHub issue
   - cline github pr

## Summary

This notebook demonstrates a complete proof-of-concept for integrating c-log-analysis with Sentry via MCP tools:

### Key Components:
1. **Anomaly Detection**: Simulated c-log-analysis output with similarity scoring
2. **Sentry Integration**: Using MCP tools instead of direct API calls
3. **Structured Event Data**: Proper Sentry event format with tags and context
4. **Complete Workflow**: From detection to resolution via Cline automation

### Benefits:
- **Automated Issue Creation**: No manual intervention needed
- **Rich Context**: Full log analysis data preserved in Sentry
- **Cross-Platform Integration**: Links Sentry issues to GitHub for tracking
- **AI-Powered Analysis**: Leverage Sentry's Seer for root cause analysis

### Next Steps:
1. Integrate this logic into c-log-analysis pipeline
2. Configure Sentry MCP server with proper authentication
3. Set up GitHub integration for automated issue creation
4. Test end-to-end workflow with real log data